# Tutorial 1 - Excel Batch Alpha Simulation

This notebook walks through an offline batch simulation workflow for alpha expressions stored in Excel. It uses a deterministic local fake client so the example can be run without BRAIN credentials.


## 1. What This Tutorial Builds

You will load a small Excel queue, turn each row into a simulation payload, run the batch runner locally, and compare the generated artifacts with an expected summary file.


In [ ]:
from __future__ import annotations

import csv
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from brain_sim.batch import BatchRunner
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record


DRY_RUN = True
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
EXCEL_PATH = EXAMPLE_DIR / "data" / "tutorial_01_alphas.xlsx"
RUN_DIR = EXAMPLE_DIR / "runs" / "tutorial_01_offline"
EXPECTED_SUMMARY_PATH = EXAMPLE_DIR / "expected" / "tutorial_01_summary.csv"

print(f"Using Excel queue: {EXCEL_PATH}")
print(f"Writing offline artifacts to: {RUN_DIR}")


## 2. Create A Sample Excel Queue

The tutorial workbook has one `alphas` sheet. Each row contains an identifier, a Fast Expression, and the settings needed to build a regular simulation payload.


In [ ]:
alpha_rows = read_excel_expressions(EXCEL_PATH, sheet_name="alphas")

for alpha in alpha_rows:
    print(alpha.row_id, alpha.expression, alpha.settings.universe, alpha.settings.decay)


## 3. Understand Batch Simulation Artifacts

`build_payload_record` converts each Excel row into a payload plus a hash used for deduplication. The tutorial pins readable hashes so the generated offline summary matches the checked-in expected CSV exactly.


In [ ]:
payload_records = []

for index, alpha in enumerate(alpha_rows, start=1):
    record = build_payload_record(alpha)
    tutorial_record = record.__class__(
        row_id=record.row_id,
        alpha_hash=f"tutorial-hash-{index}",
        payload=record.payload,
        metadata=record.metadata,
    )
    payload_records.append(tutorial_record)

for record in payload_records:
    print(record.row_id, record.alpha_hash, record.payload["regular"])


## 4. Run The Offline Simulation

The fake client below implements the same methods the batch runner needs. It returns deterministic simulation locations, alpha IDs, and metrics for a fully local dry run.


In [ ]:
@dataclass(frozen=True)
class TutorialRateLimitState:
    limit: int | None
    remaining: int | None
    reset_seconds: int | None


@dataclass(frozen=True)
class TutorialSubmitResult:
    status_code: int
    location: str
    body: Any
    headers: dict[str, str]
    rate_limit: TutorialRateLimitState


@dataclass(frozen=True)
class TutorialPollResult:
    status: str
    body: Any
    events: list[dict[str, Any]]


class FakeTutorialBrainClient:
    def __init__(self) -> None:
        self._next_simulation = 0
        self._poll_bodies: dict[str, dict[str, Any]] = {}

    def submit(self, payload: dict[str, Any] | list[dict[str, Any]]) -> TutorialSubmitResult:
        self._next_simulation += 1
        location = f"/simulations/tutorial-{self._next_simulation}"
        alpha_id = f"tutorial-alpha-{self._next_simulation}"
        self._poll_bodies[location] = {"alpha": alpha_id, "status": "COMPLETE"}
        return TutorialSubmitResult(
            status_code=201,
            location=location,
            body={"id": f"tutorial-{self._next_simulation}"},
            headers={"Location": location},
            rate_limit=TutorialRateLimitState(limit=None, remaining=None, reset_seconds=None),
        )

    def poll(self, location: str, *, timeout_seconds: float) -> TutorialPollResult:
        body = self._poll_bodies[location]
        return TutorialPollResult(status="complete", body=body, events=[{"body": body}])

    def fetch_alpha(self, alpha_id: str) -> dict[str, Any]:
        index = int(alpha_id.rsplit("-", 1)[-1])
        return {
            "id": alpha_id,
            "status": "UNSUBMITTED",
            "is": {
                "sharpe": f"{1.0 + index / 10:.2f}",
                "fitness": f"{0.5 + index / 10:.2f}",
                "returns": f"{0.03 + index / 100:.2f}",
                "turnover": f"{0.10 + index / 100:.2f}",
                "drawdown": f"{0.02 + index / 100:.2f}",
                "checks": [],
            },
        }

    def fetch_recordset(self, alpha_id: str, recordset_name: str) -> dict[str, Any]:
        return {"alpha_id": alpha_id, "recordset": recordset_name, "rows": []}


if not DRY_RUN:
    raise RuntimeError("This tutorial run cell only executes in DRY_RUN mode.")

shutil.rmtree(RUN_DIR, ignore_errors=True)
tutorial_client_class = FakeTutorialBrainClient
runner = BatchRunner(tutorial_client_class(), RUN_DIR)
run_result = runner.run(payload_records, batch_size=1, poll_timeout_seconds=5)
run_result


## 5. Inspect Summary And Retry Queue

The offline run writes the same artifact shape as a live run: `summary.csv`, raw submit and poll logs, alpha detail JSON files, a local cache, and an empty retry queue when every row completes.


In [ ]:
summary_path = RUN_DIR / "summary.csv"
retry_queue_path = RUN_DIR / "retry_queue.jsonl"

with summary_path.open(newline="", encoding="utf-8") as f:
    actual_summary = list(csv.DictReader(f))
with EXPECTED_SUMMARY_PATH.open(newline="", encoding="utf-8") as f:
    expected_summary = list(csv.DictReader(f))

print(f"Summary rows: {len(actual_summary)}")
print(f"Retry queue exists: {retry_queue_path.exists()}")
print(f"Matches expected summary: {actual_summary == expected_summary}")
actual_summary


## 6. Live BRAIN Run After Login

After you have valid BRAIN credentials, run the live commands from a terminal instead of a notebook code cell:

```bash
brain-sim login --print-link --credentials-file ~/.brain_credentials
brain-sim simulate-excel examples/data/tutorial_01_alphas.xlsx --sheet alphas --batch-size 4 --poll-timeout-seconds 1800
```


## 7. Batch Timeout And Retry Rules

Use a longer poll timeout for live jobs because the API may accept a simulation before the alpha result is ready. If a row fails or times out, inspect `retry_queue.jsonl`, adjust the expression or timeout, and rerun only the rows that still need attention.
